In [1]:
import pandas as pd

movies =pd.read_csv("movies_with_categories.csv")

In [2]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None)
classifier("I love this!")

Device set to use cpu


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419791977852583},
  {'label': 'sadness', 'score': 0.002092393347993493},
  {'label': 'disgust', 'score': 0.001611992483958602},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [3]:
classifier(movies["description"][0])

[[{'label': 'disgust', 'score': 0.45869842171669006},
  {'label': 'anger', 'score': 0.22572270035743713},
  {'label': 'neutral', 'score': 0.18345525860786438},
  {'label': 'sadness', 'score': 0.08785257488489151},
  {'label': 'fear', 'score': 0.03320051357150078},
  {'label': 'joy', 'score': 0.007174637168645859},
  {'label': 'surprise', 'score': 0.003895946079865098}]]

In [4]:
classifier(movies["description"][0].split("."))

[[{'label': 'disgust', 'score': 0.36206525564193726},
  {'label': 'anger', 'score': 0.2906533181667328},
  {'label': 'neutral', 'score': 0.15738093852996826},
  {'label': 'sadness', 'score': 0.13271403312683105},
  {'label': 'fear', 'score': 0.04200555011630058},
  {'label': 'joy', 'score': 0.009986055083572865},
  {'label': 'surprise', 'score': 0.00519487913697958}],
 [{'label': 'neutral', 'score': 0.549476683139801},
  {'label': 'sadness', 'score': 0.11169019341468811},
  {'label': 'disgust', 'score': 0.10400673747062683},
  {'label': 'surprise', 'score': 0.07876546680927277},
  {'label': 'anger', 'score': 0.06413363665342331},
  {'label': 'fear', 'score': 0.051362838596105576},
  {'label': 'joy', 'score': 0.04056441783905029}]]

In [5]:
sentences = movies["description"][0].split(".")
predictions = classifier(sentences)

In [6]:
sentences[0]

"A bored and domesticated Shrek pacts with deal-maker Rumpelstiltskin to get back to feeling like a real ogre again, but when he's duped and sent to a twisted version of Far Far Away—where Rumpelstiltskin is king, ogres are hunted, and he and Fiona have never met—he sets out to restore his world and reclaim his true love"

In [7]:
predictions[0]

[{'label': 'disgust', 'score': 0.36206525564193726},
 {'label': 'anger', 'score': 0.2906533181667328},
 {'label': 'neutral', 'score': 0.15738093852996826},
 {'label': 'sadness', 'score': 0.13271403312683105},
 {'label': 'fear', 'score': 0.04200555011630058},
 {'label': 'joy', 'score': 0.009986055083572865},
 {'label': 'surprise', 'score': 0.00519487913697958}]

In [8]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.2906533181667328},
 {'label': 'disgust', 'score': 0.36206525564193726},
 {'label': 'fear', 'score': 0.04200555011630058},
 {'label': 'joy', 'score': 0.009986055083572865},
 {'label': 'neutral', 'score': 0.15738093852996826},
 {'label': 'sadness', 'score': 0.13271403312683105},
 {'label': 'surprise', 'score': 0.00519487913697958}]

In [9]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
show_id = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [10]:
for i in range(10):
    show_id.append(movies["show_id"][i])
    sentences = movies["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [11]:
emotion_scores

{'anger': [np.float64(0.2906533181667328),
  np.float64(0.0854841023683548),
  np.float64(0.06413363665342331),
  np.float64(0.07691820710897446),
  np.float64(0.23342791199684143),
  np.float64(0.06413363665342331),
  np.float64(0.1992855817079544),
  np.float64(0.06413363665342331),
  np.float64(0.06413363665342331),
  np.float64(0.07600753754377365)],
 'disgust': [np.float64(0.36206525564193726),
  np.float64(0.10400673747062683),
  np.float64(0.10400673747062683),
  np.float64(0.10400673747062683),
  np.float64(0.3062315583229065),
  np.float64(0.10403338819742203),
  np.float64(0.10400673747062683),
  np.float64(0.941213846206665),
  np.float64(0.10400673747062683),
  np.float64(0.10400673747062683)],
 'fear': [np.float64(0.051362838596105576),
  np.float64(0.051362838596105576),
  np.float64(0.9276933670043945),
  np.float64(0.051362838596105576),
  np.float64(0.9662854671478271),
  np.float64(0.9092564582824707),
  np.float64(0.7897287011146545),
  np.float64(0.05136283859610557

In [12]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
show_id = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(movies))):
    show_id.append(movies["show_id"][i])
    sentences = movies["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 7199/7199 [28:34<00:00,  4.20it/s] 


In [13]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["show_id"] = show_id

In [14]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,show_id
0,0.290653,0.362065,0.051363,0.040564,0.549477,0.132714,0.078765,10192
1,0.085484,0.104007,0.051363,0.040564,0.752492,0.111690,0.078765,27205
2,0.064134,0.104007,0.927693,0.109839,0.549477,0.111690,0.078765,12444
3,0.076918,0.104007,0.051363,0.642960,0.549477,0.609708,0.078765,38757
4,0.233428,0.306232,0.966285,0.040564,0.549477,0.111690,0.078765,10191
...,...,...,...,...,...,...,...,...
7194,0.064134,0.104007,0.051363,0.040564,0.893861,0.111690,0.078765,842924
7195,0.064134,0.709687,0.173097,0.700117,0.549477,0.653512,0.078765,1352077
7196,0.390935,0.127391,0.441361,0.040564,0.549477,0.111690,0.078765,934456
7197,0.064134,0.104007,0.176641,0.040564,0.549477,0.680627,0.078765,1254780


In [15]:
movies = pd.merge(movies, emotions_df, on = "show_id")

In [16]:
movies

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,simple_category,show_id1,predicted_category,anger,disgust,fear,joy,sadness,surprise,neutral
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,...,Fiction,10192.0,Fiction,0.290653,0.362065,0.051363,0.040564,0.549477,0.132714,0.078765
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,...,Nonfiction,27205.0,Nonfiction,0.085484,0.104007,0.051363,0.040564,0.752492,0.111690,0.078765
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,...,Nonfiction,12444.0,Nonfiction,0.064134,0.104007,0.927693,0.109839,0.549477,0.111690,0.078765
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,...,Fiction,38757.0,Fiction,0.076918,0.104007,0.051363,0.642960,0.549477,0.609708,0.078765
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,...,Nonfiction,10191.0,Nonfiction,0.233428,0.306232,0.966285,0.040564,0.549477,0.111690,0.078765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7194,842924,Movie,The Life of Chuck,Mike Flanagan,"Tom Hiddleston, Mark Hamill, Chiwetel Ejiofor,...",United States of America,2025-05-30,2025,0.000,NaN,...,Nonfiction,842924.0,Nonfiction,0.064134,0.104007,0.051363,0.040564,0.893861,0.111690,0.078765
7195,1352077,Movie,Verhängnisvolle Leidenschaft Sylt,Elmar Fischer,"Cornelia Gröschel, Artjom Gilz, Franz Dinda, L...",Germany,2025-01-31,2025,0.000,NaN,...,Fiction,1352077.0,Fiction,0.064134,0.709687,0.173097,0.700117,0.549477,0.653512,0.078765
7196,934456,Movie,Americana,Tony Tost,"Sydney Sweeney, Paul Walter Hauser, Halsey, Si...","Canada, United States of America",2025-08-22,2025,0.000,NaN,...,Nonfiction,934456.0,Nonfiction,0.390935,0.127391,0.441361,0.040564,0.549477,0.111690,0.078765
7197,1254780,Movie,The Grove,"Acoryé White, Patrycja Kępa","Acoryé White, Carl Anthony Payne II, Psalms Sa...",United States of America,2025-04-04,2025,0.000,NaN,...,Fiction,1254780.0,Fiction,0.064134,0.104007,0.176641,0.040564,0.549477,0.680627,0.078765


In [17]:
movies.to_csv("movies_with_emotions.csv", index = False)